In [15]:
#LOAD DATASET
import pandas as pd
from google.colab import drive
import os

drive.mount('/content/drive')

print("Searching for nlp.csv in /content/drive/...")
found_path = None
for root, dirs, files in os.walk('/content/drive/'):
    if 'nlp.csv' in files:
        found_path = os.path.join(root, 'nlp.csv')
        break

if found_path:
    print(f"File found at: {found_path}")
    df = pd.read_csv(found_path)
    print("CSV file loaded successfully!")
    display(df.head())
else:
    print("Could not find nlp.csv anywhere in your Google Drive.")
    print("Checking common directories...")
    my_drive_path = '/content/drive/My Drive/'
    if os.path.exists(my_drive_path):
        print(f"Folders in My Drive: {[d for d in os.listdir(my_drive_path) if os.path.isdir(os.path.join(my_drive_path, d))]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Searching for nlp.csv in /content/drive/...
File found at: /content/drive/MyDrive/Kaggle_Data_Backup/nlp.csv
CSV file loaded successfully!


,text_description,source
0,While removing the drill rod of the Jumbo 08 f...,industrial_safety
1,During the activation of a sodium sulphide pum...,industrial_safety
2,In the sub-station MILPO located at level +170...,industrial_safety
3,Being 9:45 am. approximately in the Nv. 1880 C...,industrial_safety
4,Approximately at 11:45 a.m. in circumstances t...,industrial_safety


In [2]:
#PRINT COLUMN
print(df.columns)

Index(['text_description', 'source'], dtype='object')


In [3]:
display(df.head(10))

,text_description,source
0,While removing the drill rod of the Jumbo 08 f...,industrial_safety
1,During the activation of a sodium sulphide pum...,industrial_safety
2,In the sub-station MILPO located at level +170...,industrial_safety
3,Being 9:45 am. approximately in the Nv. 1880 C...,industrial_safety
4,Approximately at 11:45 a.m. in circumstances t...,industrial_safety
5,During the unloading operation of the ustulado...,industrial_safety
6,The collaborator reports that he was on street...,industrial_safety
7,"At approximately 04:50 p.m., when the mechanic...",industrial_safety
8,Employee was sitting in the resting area at le...,industrial_safety
9,At the moment the forklift operator went to ma...,industrial_safety


In [4]:
#TRAIN MODEL
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import joblib

X_train, X_test, y_train, y_test = train_test_split(
    df['text_description'],
    df['source'],
    test_size=0.2,
    stratify=df['source'],
    random_state=42
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3,5)
    )),

    ('clf', SGDClassifier(
        loss='hinge',
        class_weight='balanced',
        random_state=42
    ))
])

# Training
print("Training model...")
pipeline.fit(X_train, y_train)

print("Training selesai!")

Training model...
Training selesai!


In [5]:
#ACCURACY
y_pred = pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print("Accuracy:", acc)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 1.0

Classification Report:
                   precision    recall  f1-score   support

industrial_safety       1.00      1.00      1.00        85
             osha       1.00      1.00      1.00       970

         accuracy                           1.00      1055
        macro avg       1.00      1.00      1.00      1055
     weighted avg       1.00      1.00      1.00      1055



In [16]:
#ACCURACY
train_acc = pipeline.score(X_train, y_train)
test_acc = pipeline.score(X_test, y_test)

print("Train Accuracy:", train_acc)
print("Test Accuracy :", test_acc)

Train Accuracy: 1.0
Test Accuracy : 1.0


In [6]:
#SAVE MODEL
joblib.dump(pipeline, 'safety_model.joblib')

print("Model berhasil disimpan!")

Model berhasil disimpan!


In [8]:
#LOAD FOR TEST MODEL
import os

print(os.listdir())

['.config', 'drive', 'safety_model.joblib', 'sample_data']


In [18]:
#TEST MODEL
import joblib

loaded_model = joblib.load('safety_model.joblib')

print("Model berhasil diload!")
print(type(loaded_model))

texts = [
    "Worker had bad aid during welding process",
    "Employee fell from ladder",
    "Chemical exposure in factory"
]

predictions = loaded_model.predict(texts)

for text, pred in zip(texts, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {pred}")
    print("-" * 30)

['.config', 'drive', 'safety_model.joblib', 'sample_data']
Model berhasil diload!
<class 'sklearn.pipeline.Pipeline'>
Text: Worker had bad aid during welding process
Prediction: osha
------------------------------
Text: Employee fell from ladder
Prediction: osha
------------------------------
Text: Chemical exposure in factory
Prediction: industrial_safety
------------------------------


In [ ]:
#TEMP
print(f"Jumlah baris: {df.shape[0]}")
print(f"Jumlah kolom: {df.shape[1]}")

print("\nInformasi Struktur Data:")
df.info()

print("\nDeskripsi Jenis Data:")
display(df.describe(include='all'))